<a href="https://colab.research.google.com/github/Jayku88/22AIE301_Probabilistic_Reasoning/blob/main/22AIE301_Lab_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 22AIE301 - Probabilistic Reasoning
## Lab 06 - Graphical Models: d-Separation, I-Maps, MRFs, Factor Graphs & CRFs

| | |
|---|---|
| **Name** | ADITHYAN ABHILASH SAJU |
| **Roll No** | AM.SC.U4AIE24006 |
| **Date** | 05-07-2026 |
| **Lab Slot** | 06 |


**Learning Objectives**
- Identify the three primitive node patterns (cascade, fork, v-structure) and their independence behavior.
- Implement the four-step d-separation recipe (ancestral graph → moralize → disorient → delete givens) from scratch, and cross-verify against `pgmpy`.
- Distinguish an **I-map** from a **minimal I-map**.
- Read an undirected graph: identify neighbors, cliques, and maximal cliques.
- Write the Gibbs form $p(x) = \frac{1}{Z}\prod_c \phi_c(x_c)$ and compute $Z$ by brute force.
- Compute Markov blankets and cutsets on an undirected graph.
- Build a factor graph and understand why it removes clique-scope ambiguity.
- Compute $Z(x)$ for a Conditional Random Field and see why it depends on the input.

**Instructions**
- Fill in every `___` blank. Run top to bottom — every section ends with `assert` checks.
- Your numbers are derived from your roll number via `np.random.seed(ROLL_NO)`, so your graphs/values will differ from your neighbor's.


## 0. Setup

In [1]:
!pip install pgmpy -q
import numpy as np
import networkx as nx
import itertools
import matplotlib.pyplot as plt
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

ROLL_NO = 6    # TODO: enter the numeric part of your roll number, e.g. 42
np.random.seed(ROLL_NO)
print("Seeded with ROLL_NO =", ROLL_NO)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.4/165.4 kB 6.3 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


Seeded with ROLL_NO = 6



# Part A - d-Separation and I-Maps



## 1. The Building Blocks - Cascade, Fork, V-structure

For each pattern below, build a 3-node Bayesian network in `pgmpy` with **randomly generated CPDs**
(so the specific numbers are yours), then check numerically whether $X$ and $Y$ are independent - both
when the middle node $Z$ is unobserved and when it is observed - by comparing the **joint** $P(X,Y\mid \cdot)$
against the **product of marginals** $P(X\mid\cdot)\,P(Y\mid\cdot)$.


In [5]:
def random_cpd(var, evidence, evidence_card, card=2):
    if not evidence: # No evidence variables, it's a root node
        p1 = np.random.uniform(0.2, 0.8)
        return TabularCPD(var, card, [[1 - p1], [p1]])
    else: # Has evidence variables
        n_cols = int(np.prod(evidence_card))
        p1_values = np.random.uniform(0.2, 0.8, size=n_cols)
        table = [(1 - p1_values).tolist(), p1_values.tolist()] # TODO: second row is P(var=1|...) = p1
        return TabularCPD(var, card, table, evidence=evidence, evidence_card=evidence_card)

def independent_given(model, X, Y, evidence=None, tol=1e-6):
    infer = VariableElimination(model)
    joint = infer.query([X, Y], evidence=evidence, show_progress=False)
    pX = infer.query([X], evidence=evidence, show_progress=False)
    pY = infer.query([Y], evidence=evidence, show_progress=False)
    product = np.outer(pX.values, pY.values)
    return np.allclose(joint.values, product, atol=tol)      # TODO: compare joint.values against `product`

In [ ]:

# --- 1a. Cascade: X -> Z -> Y ---
cascade = DiscreteBayesianNetwork([('X', 'Z'), ___])      # TODO: add the edge Z -> Y
cpd_x = random_cpd('X', [], [])
cpd_z = random_cpd('Z', ['X'], [2])
cpd_y = random_cpd('Y', [___], [2])                        # TODO: Y's parent
cascade.add_cpds(cpd_x, cpd_z, cpd_y)
assert cascade.check_model()

cascade_dep_unobserved = not independent_given(cascade, 'X', 'Y', evidence=None)
cascade_indep_observed = independent_given(cascade, 'X', 'Y', evidence={'Z': 0})

assert cascade_dep_unobserved == True,  "Cascade: X,Y should be DEPENDENT when Z is unobserved"
assert cascade_indep_observed == True,  "Cascade: X,Y should be INDEPENDENT when Z is observed"
print("Cascade pattern verified.")


In [ ]:

# --- 1b. Fork: X <- Z -> Y ---
fork = DiscreteBayesianNetwork([('Z', 'X'), (___, ___)])   # TODO: Z -> Y
cpd_z2 = random_cpd('Z', [], [])
cpd_x2 = random_cpd('X', ['Z'], [2])
cpd_y2 = random_cpd('Y', ['Z'], [2])
fork.add_cpds(cpd_z2, cpd_x2, cpd_y2)
assert fork.check_model()

fork_dep_unobserved = not independent_given(fork, 'X', 'Y', evidence=None)
fork_indep_observed = independent_given(fork, 'X', 'Y', evidence={___: 0})  # TODO: condition on Z

assert fork_dep_unobserved == True
assert fork_indep_observed == True
print("Fork pattern verified.")


In [ ]:

# --- 1c. V-structure (collider): X -> Z <- Y ---
vstruct = DiscreteBayesianNetwork([('X', 'Z'), ('Y', 'Z')])
cpd_x3 = random_cpd('X', [], [])
cpd_y3 = random_cpd('Y', [], [])
cpd_z3 = random_cpd('Z', ['X', 'Y'], [2, 2])
vstruct.add_cpds(cpd_x3, cpd_y3, cpd_z3)
assert vstruct.check_model()

vstruct_indep_unobserved = independent_given(vstruct, 'X', 'Y', evidence=None)
vstruct_dep_observed = not independent_given(vstruct, ___, ___, evidence={'Z': 0})  # TODO: fill 'X','Y'

assert vstruct_indep_unobserved == True, "V-structure: X,Y should be INDEPENDENT when Z is unobserved"
assert vstruct_dep_observed == True,     "V-structure: X,Y should become DEPENDENT once Z is observed"
print("V-structure pattern verified.")



## 2. Implementing d-separation From Scratch

Implement the four-step recipe as a function `is_d_separated(G, X, Y, Z)` operating on a `networkx.DiGraph`.

1. **Ancestral graph** - keep $X$, $Y$, $\mathbf{Z}$, and all their ancestors.
2. **Moralize** - connect every pair of nodes sharing a child.
3. **Disorient** - drop all arrowheads.
4. **Delete givens** - remove every node in $\mathbf{Z}$.

Then $X,Y$ are d-separated given $\mathbf{Z}$ iff they are **disconnected** in the resulting graph.


In [ ]:

def ancestors_of(G, nodes):
    \"\"\"Return `nodes` unioned with all of their ancestors in the DAG G.\"\"\"
    anc = set(nodes)
    for n in nodes:
        anc |= ___                          # TODO: nx.ancestors(G, n)
    return anc

def moralize(G, keep_nodes):
    \"\"\"Return the undirected moral graph of G, restricted to `keep_nodes`.\"\"\"
    H = nx.Graph()
    H.add_nodes_from(keep_nodes)
    for n in keep_nodes:
        parents = [p for p in G.predecessors(n) if p in keep_nodes]
        for p in parents:
            H.add_edge(p, n)                              # parent-child edge
        for p1, p2 in itertools.combinations(parents, 2):
            H.add_edge(___, ___)                          # TODO: marry co-parents p1, p2
    return H

def is_d_separated(G, X, Y, Z):
    \"\"\"Return True iff X and Y are d-separated given evidence set Z in DAG G.\"\"\"
    Z = set(Z)
    relevant = ancestors_of(G, {X, Y} | Z)     # step 1: ancestral graph
    H = moralize(G, relevant)                  # step 2 (+3, undirected already)
    H.remove_nodes_from(___)                   # TODO: step 4 — delete the givens
    if X not in H or Y not in H:
        return True
    return not nx.has_path(H, X, Y)            # disconnected => d-separated



## 3. Cross-Verification on the Lecture's Running Example

Network: $A\to C \leftarrow B$, $C\to D$, $C\to E$, $D\to F\to G$ (same as the slides).

For each query below, first write down **your predicted answer** (`True` = d-separated / independent),
then let both your own `is_d_separated` and `pgmpy`'s `is_dconnected` check it. All three must agree.


In [ ]:

G_dag = nx.DiGraph()
G_dag.add_edges_from([('A', 'C'), ('B', 'C'), ('C', 'D'), ('C', 'E'), ('D', 'F'), ('F', 'G')])

bn = DiscreteBayesianNetwork([('A', 'C'), ('B', 'C'), ('C', 'D'), ('C', 'E'), ('D', 'F'), ('F', 'G')])

queries = [
    # (X, Y, Z, your_predicted_answer)
    ('A', 'B', {'D', 'F'}, ___),   # Q1 from the slides
    ('A', 'B', set(),      ___),   # Q2
    ('A', 'B', {'C'},      ___),   # Q3
    ('D', 'E', {'C'},      ___),   # Q4
    ('D', 'E', set(),      ___),   # Q5
    ('D', 'E', {'A', 'B'}, ___),   # Q6
]

for X, Y, Z, predicted in queries:
    mine = is_d_separated(G_dag, X, Y, Z)
    pgmpy_says = not bn.is_dconnected(X, Y, observed=list(Z))
    assert predicted == mine == pgmpy_says, \
        f"Mismatch on {X} _|_ {Y} | {sorted(Z)}: predicted={predicted}, yours={mine}, pgmpy={pgmpy_says}"
    print(f"{X} _|_ {Y} | {sorted(Z) if Z else '{}'}  ->  {mine}   (matches pgmpy)")



## 4. Your Own Random Query

A random DAG on 7 nodes has been generated using your `ROLL_NO` as seed. Pick a query of your own
(any two non-adjacent nodes and any evidence set), predict the answer by tracing the four-step recipe
**by hand on paper**, then confirm it with your code.


In [ ]:

def random_dag(n_nodes=7, edge_prob=0.35):
    nodes = list("ABCDEFG")[:n_nodes]
    G = nx.DiGraph()
    G.add_nodes_from(nodes)
    for i, u in enumerate(nodes):
        for v in nodes[i+1:]:
            if np.random.rand() < edge_prob:
                G.add_edge(u, v)      # edges only go from earlier to later letters => guaranteed acyclic
    return G

my_G = random_dag()
print("Your random DAG edges:", list(my_G.edges()))

my_bn = DiscreteBayesianNetwork(list(my_G.edges()))

# TODO: pick your own query - X, Y should be two non-adjacent nodes in my_G; Z any subset of the rest
my_X = ___
my_Y = ___
my_Z = ___    # e.g. set() or {'C'}

my_prediction = ___     # TODO: True or False, from tracing the recipe by hand

mine = is_d_separated(my_G, my_X, my_Y, my_Z)
pgmpy_says = not my_bn.is_dconnected(my_X, my_Y, observed=list(my_Z))
assert mine == pgmpy_says, "Your implementation disagrees with pgmpy on your own graph — debug is_d_separated."
assert my_prediction == mine, "Your hand-traced prediction does not match the algorithm's answer — retrace the 4 steps."
print(f"Your query {my_X} _|_ {my_Y} | {my_Z}  ->  {mine}  (matches pgmpy)")



## 5. I-Maps and Minimal I-Maps

A distribution $P(A,B,C)$ has been generated (seeded by your roll number) with **exactly one**
nontrivial independence: $A \perp C \mid B$ - all other pairs, marginally or conditionally, are dependent.

You will test four **candidate** graphs against this fixed $P$ and classify each as:
- **not an I-map** (claims a false independence),
- **I-map, minimal** (every claim is true, and removing any edge would break that), or
- **I-map, not minimal** (every claim is true, but it has redundant edges).


In [ ]:

# --- Build the distribution P(A,B,C) with exactly one nontrivial independence: A _|_ C | B ---
pA = np.random.uniform(0.3, 0.7)
pB_given_A = np.random.uniform(0.2, 0.8, size=2)   # P(B=1 | A=a)
pC_given_B = np.random.uniform(0.2, 0.8, size=2)   # P(C=1 | B=b)

P = {}
for a, b, c in itertools.product([0, 1], repeat=3):
    pa = pA if a == 1 else 1 - pA
    pb = pB_given_A[a] if b == 1 else 1 - pB_given_A[a]
    pc = pC_given_B[b] if c == 1 else 1 - pC_given_B[b]
    P[(a, b, c)] = pa * pb * ___                    # TODO: multiply in pc

assert abs(sum(P.values()) - 1.0) < 1e-9, "P must sum to 1"


In [ ]:

def marg_prob(P, assignment):
    \"\"\"assignment: dict of a subset of {'A','B','C'} -> value. Returns P(assignment), summed over the rest.\"\"\"
    total = 0.0
    for (a, b, c), p in P.items():
        full = {'A': a, 'B': b, 'C': c}
        if all(full[k] == v for k, v in assignment.items()):
            total += p
    return total

def cond_indep(P, X, Y, Z, tol=1e-6):
    \"\"\"Numerically test X _|_ Y | Z against the table P. Z is a list of variable names.\"\"\"
    dom = {'A': [0, 1], 'B': [0, 1], 'C': [0, 1]}
    for zvals in itertools.product(*[dom[z] for z in Z]):
        zassign = dict(zip(Z, zvals))
        pz = marg_prob(P, zassign)
        if pz < 1e-12:
            continue
        for xval in dom[X]:
            for yval in dom[Y]:
                pxyz = marg_prob(P, {**zassign, X: xval, Y: yval})
                pxz = marg_prob(P, {**zassign, X: xval})
                pyz = marg_prob(P, {**zassign, Y: yval})
                lhs = pxyz / pz
                rhs = (pxz / pz) * (___ / pz)        # TODO: pyz
                if abs(lhs - rhs) > tol:
                    return False
    return True

# Sanity check against the known structure of P
assert cond_indep(P, 'A', 'C', ['B']) == True,  "By construction, A _|_ C | B should hold"
assert cond_indep(P, 'A', 'B', [])    == False
assert cond_indep(P, 'B', 'C', [])    == False
assert cond_indep(P, 'A', 'C', [])    == False
print("Ground-truth independence structure of P confirmed: only A _|_ C | B holds.")


In [ ]:

candidates = {
    'empty':        nx.DiGraph([]),
    'chain':        nx.DiGraph([('A', 'B'), ('B', 'C')]),
    'v-structure':  nx.DiGraph([('A', 'B'), ('C', 'B')]),
    'complete':     nx.DiGraph([('A', 'B'), ('B', 'C'), ('A', 'C')]),
}
for g in candidates.values():
    g.add_nodes_from(['A', 'B', 'C'])

all_pairs_and_seps = [
    ('A', 'C', ['B']),
    ('A', 'B', []),
    ('B', 'C', []),
    ('A', 'C', []),
]

verdicts = {}
for name, cand in candidates.items():
    is_imap = True
    for X, Y, Z in all_pairs_and_seps:
        graph_claims_indep = is_d_separated(cand, X, Y, set(Z))
        if graph_claims_indep:
            # An I-map may ONLY claim independencies that are actually true
            actually_indep = cond_indep(P, X, Y, Z)
            if not actually_indep:
                is_imap = ___              # TODO: False — a false claim disqualifies it
                break
    verdicts[name] = is_imap
    print(f"{name:12s} edges={list(cand.edges())!s:35s} I-map? {is_imap}")

assert verdicts['empty'] == False,       "Empty graph claims independencies that are false here"
assert verdicts['chain'] == True,        "Chain A->B->C claims only A _|_ C | B, which is true"
assert verdicts['v-structure'] == False, "V-structure wrongly claims A _|_ C marginally"
assert verdicts['complete'] == True,     "Complete graph claims nothing, so it's trivially an I-map"


In [ ]:

# A minimal I-map is an I-map from which no edge can be removed without breaking the I-map property.
def is_imap_check(cand, P, all_pairs_and_seps):
    for X, Y, Z in all_pairs_and_seps:
        if is_d_separated(cand, X, Y, set(Z)) and not cond_indep(P, X, Y, Z):
            return False
    return True

def is_minimal_imap(G, P, all_pairs_and_seps):
    if not is_imap_check(G, P, all_pairs_and_seps):
        return False
    for u, v in list(G.edges()):
        H = G.copy()
        H.remove_edge(u, v)
        if is_imap_check(H, P, ___):                     # TODO: pass all_pairs_and_seps
            return False    # found a removable edge => not minimal
    return True

for name, cand in candidates.items():
    minimal = is_minimal_imap(cand, P, all_pairs_and_seps) if verdicts[name] else False
    label = "minimal I-map" if minimal else ("I-map, NOT minimal" if verdicts[name] else "not an I-map")
    print(f"{name:12s} -> {label}")

assert is_minimal_imap(candidates['chain'], P, all_pairs_and_seps) == True
assert is_minimal_imap(candidates['complete'], P, all_pairs_and_seps) == False



# Part B - Markov Random Fields, Factor Graphs, and CRFs



## 6. Reading an Undirected Graph - Neighbors and Cliques

Build the 4-cycle graph $A$–$B$–$C$–$D$–$A$ with `networkx`, then write a brute-force maximal-clique
finder and check it against `networkx.find_cliques`.


In [ ]:

UG = nx.Graph()
UG.add_edges_from([('A', 'B'), ('B', 'C'), ('C', 'D'), (___, ___)])   # TODO: close the cycle, D-A

neighbors_A = set(UG.neighbors('A'))
assert neighbors_A == {'B', 'D'}, "In a 4-cycle A-B-C-D-A, A's neighbors should be B and D"
print("ne(A) =", neighbors_A)


In [ ]:

def is_clique(G, node_set):
    for u, v in itertools.combinations(node_set, 2):
        if not G.has_edge(u, v):
            return ___                 # TODO: False
    return True

def all_maximal_cliques(G):
    nodes = list(G.nodes())
    cliques = []
    for r in range(1, len(nodes) + 1):
        for subset in itertools.combinations(nodes, r):
            if is_clique(G, subset):
                cliques.append(frozenset(subset))
    maximal = [c for c in cliques if not any(c < other for other in cliques)]
    return maximal

my_cliques = sorted(all_maximal_cliques(UG), key=lambda s: sorted(s))
nx_cliques = sorted((frozenset(c) for c in nx.find_cliques(UG)), key=lambda s: sorted(s))

assert my_cliques == nx_cliques, f"Mismatch: yours={my_cliques} vs networkx={nx_cliques}"
print("Maximal cliques:", my_cliques)



## 7. Markov Blanket and Cutsets

For an **undirected** graph, $\mathrm{MB}(X) = \mathrm{ne}(X)$ — no "spouses" step needed (unlike a
Bayesian network). A **cutset** $S$ is a set of nodes whose removal disconnects the graph; if $S$ separates
$U$ from $W$, then $U \perp W \mid S$.


In [ ]:

def markov_blanket(G, node):
    return set(G.neighbors(node))

mb_A = markov_blanket(UG, 'A')
assert mb_A == {'B', 'D'}
print("MB(A) =", mb_A, " => A _|_ C | {B, D} in the 4-cycle")


In [ ]:

# Two triangles joined through a single hub node 'g'
H = nx.Graph()
H.add_edges_from([('A', 'B'), ('A', 'C'), ('B', 'g'), ('C', 'g'),
                   ('g', 'D'), ('g', 'E'), ('D', 'F'), ('E', 'F')])

def is_cutset(H, S):
    \"\"\"True iff removing S disconnects H into 2+ components.\"\"\"
    H2 = H.copy()
    H2.remove_nodes_from(S)
    return nx.number_connected_components(H2) > ___     # TODO: 1

assert is_cutset(H, {'g'}) == True, "{'g'} should separate {A,B,C} from {D,E,F}"

H_minus_g = H.copy()
H_minus_g.remove_node('g')
components = list(nx.connected_components(H_minus_g))
print("Components after removing 'g':", components)
assert len(components) == 2



## 8. Reflection

**A.** Propose your own example (from a **non-technical** domain, of your choosing) of a distribution that
has **no perfect map** as a DAG - i.e., no directed graph captures *exactly* its independence structure,
only an over- or under-approximation. Explain briefly why a DAG can't capture it exactly.

**Your answer:**

___

**B.** Pick one applied use case (e.g., image denoising with an Ising-style MRF, or part-of-speech tagging
with a linear-chain CRF). Describe how the potential functions would be designed for that task, and
connect it back to the "neighbors like to agree" / emission-transition ideas used above.

**Your answer:**

___


---
## Submission Checklist
-  Your **Roll No** is set correctly in the Setup cell (`ROLL`).
-  Save the notebook: **File → Save** (or Ctrl+S).

**Rename the file as:** `Lab06_<YourRollNo>.ipynb` and convert to PDF before submitting.